# Этап 7 V1 — TabM поверх бустинговых моделей

## Исследовательский вопрос

**Может ли TabM дать дополнительную прогностическую ценность, если использовать её не как самостоятельную модель, а поверх прогнозов сильных бустинговых моделей CatBoost, XGBoost и LightGBM?**

### Почему проверяем это сейчас?

На предыдущем этапе самостоятельная TabM была проверена на тех же 47 разрешённых признаках и показала результат хуже XGBoost:

- `Gini = 0.781310`;
- `PR-AUC = 0.567993`;
- `Recall = 0.280850`;
- `F1 = 0.406804`.

Это показало, что заменять сильную GBDT-модель на самостоятельную TabM оснований нет.

Однако это ещё не отвечало на другой вопрос: **может ли TabM быть полезна как модель второго уровня**, если вместе с исходными признаками передать ей прогнозы нескольких сильных бустинговых моделей.

### Что меняется?

Вход TabM расширяется:

- 47 исходных разрешённых признаков;
- вероятность CatBoost;
- вероятность XGBoost;
- вероятность LightGBM.

Всего TabM получает `50` входных признаков.

Вероятности GBDT строятся по leakage-safe схеме: каждая строка получает прогноз только от модели, которая не обучалась на этой строке.

### Что остаётся неизменным?

- датасет `Data_final.xlsb`;
- целевая переменная `DefMark`;
- идентификатор `INN`;
- те же 47 разрешённых исходных признаков;
- `Q_B1_norm` и `Q_B2_norm` не используются как predictors;
- внешний `StratifiedKFold` на 3 фолда;
- seed `42`;
- seeds внешних фолдов `43 / 44 / 45`;
- конфигурация TabM предыдущего этапа, кроме технически необходимого увеличения входа с 47 до 50 признаков;
- базовый протокол обучения GBDT;
- метрики оценки;
- порог `0.5` используется только как диагностический;
- final test остаётся закрытым.

Никакой подбор весов ансамбля, дополнительный tuning, balancing, calibration или threshold optimization в этом эксперименте не выполняются.

## 1. Фиксируем контракт эксперимента

### Что проверяем?

Перед построением stacking-модели фиксируем:

- контрольную сумму датасета;
- рабочую выборку;
- порядок 47 разрешённых признаков;
- результаты Stage 1 для GBDT;
- сохранённые OOF-прогнозы Stage 3;
- конфигурацию TabM;
- внешний 3-fold протокол.

### Зачем?

Stacking особенно чувствителен к утечке данных.

Если TabM получит вероятность от модели, которая уже обучалась на той же строке, качество будет искусственно завышено и эксперимент потеряет смысл.

### Как это отвечает на исследовательский вопрос?

Мы заранее фиксируем способ получения трёх новых признаков и не позволяем результату TabM влиять на их построение.

Так сравнение показывает именно дополнительную ценность гибридной схемы, а не эффект утечки или последующего подбора протокола.

### Что остаётся неизменным?

Данные, target, исходный набор из 47 признаков, folds, seeds, GBDT-протокол и правила оценки не меняются.

In [9]:
from __future__ import annotations
import hashlib, json, os, random, time, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tabm
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier
from IPython.display import HTML, Markdown, display
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, TensorDataset

def корень():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("Не найден корень проекта.")
def sha(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for b in iter(lambda:f.read(1048576),b""): h.update(b)
    return h.hexdigest()
def idx_sha(x): return hashlib.sha256(np.asarray(x,dtype=np.int64).tobytes()).hexdigest()
def metrics(y,p):
    q=(np.asarray(p)>=.5).astype(int); y=np.asarray(y,dtype=int); a=float(roc_auc_score(y,p)); tn,fp,fn,tp=confusion_matrix(y,q,labels=[0,1]).ravel()
    return {"ROC-AUC":a,"Gini":2*a-1,"PR-AUC":float(average_precision_score(y,p)),"Precision":float(precision_score(y,q,zero_division=0)),"Recall":float(recall_score(y,q,zero_division=0)),"F1":float(f1_score(y,q,zero_division=0)),"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp)}
R=корень(); G=R/"reports"/"generated"; S=R/"reports"/"summary"; DATA=R/"data"/"raw"/"Data_final.xlsb"
ST1=G/"stage1_baseline_results_V2.json"; ST3=G/"stage3_oof_predictions_V1.npz"; ST3J=G/"stage3_error_analysis_results_V1.json"
RESULT=G/"stage7_tabm_stacking_results_V1.json"; OOF=G/"stage7_tabm_stacking_oof_V1.npz"; SUMMARY=S/"stage7_tabm_stacking_summary_V1.json"; CKPT=G/"stage7_tabm_stacking_checkpoint_V1.pt"
DATA_SHA="fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930"; WORK_SHA="80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45"
TARGET,IDENTIFIER="DefMark","INN"; FORBIDDEN=["Q_B1_norm","Q_B2_norm"]; SEED=42; FSEED={1:43,2:44,3:45}
stage1=json.loads(ST1.read_text(encoding="utf-8")); FEATURES=stage1["допустимые_признаки"]; GBDT=stage1["параметры_моделей"]
CFG={"arch_type":"tabm","k":8,"n_blocks":3,"d_block":512,"activation":"ReLU","dropout":.1,"start_scaling_init":"random-signs","d_out":2,"lr":.002,"weight_decay":.0003,"betas":(.9,.999),"eps":1e-8,"batch_size":256,"max_epochs":100,"patience":16,"gradient_clip_global_norm":1.}
if len(FEATURES)!=47 or any(x in FEATURES for x in FORBIDDEN): raise RuntimeError("Нарушен контракт 47 признаков.")

## 2. Проверяем данные и происхождение OOF-прогнозов

### Что проверяем?

Перед обучением подтверждаем, что:

- используется именно зафиксированный `Data_final.xlsb`;
- рабочая выборка содержит `289 614` наблюдений;
- порядок 47 признаков не изменился;
- `Q_B1_norm` и `Q_B2_norm` отсутствуют среди predictors;
- сохранённые OOF-прогнозы CatBoost, XGBoost и LightGBM относятся к той же рабочей выборке;
- их метрики воспроизводят принятые результаты Stage 1;
- final test не использовался.

### Зачем?

Новые признаки TabM — это вероятности других моделей. Поэтому особенно важно доказать их происхождение и соответствие текущему split.

### Как это отвечает на исследовательский вопрос?

Только после этой проверки можно считать три GBDT-вероятности допустимыми признаками для следующего уровня stacking.

### Что остаётся неизменным?

Ни одна модель здесь ещё не обучается и экспериментальный протокол не меняется.

In [2]:
if sha(DATA)!=DATA_SHA: raise RuntimeError("Неверная контрольная сумма данных.")
data=pd.read_excel(DATA,engine="pyxlsb",sheet_name="Data_final").reset_index(drop=True)
if [c for c in data.columns if c not in [IDENTIFIER,TARGET,*FORBIDDEN]]!=FEATURES: raise RuntimeError("Изменён порядок разрешённых признаков.")
all_i=np.arange(len(data)); work_i,_=train_test_split(all_i,test_size=.2,stratify=data[TARGET],random_state=SEED)
if len(work_i)!=289614 or idx_sha(work_i)!=WORK_SHA: raise RuntimeError("Working sample не совпал с accepted protocol.")
X=data.loc[work_i,FEATURES].to_numpy(np.float32); y=data.loc[work_i,TARGET].to_numpy(np.int8); del data
with np.load(ST3,allow_pickle=False) as z:
    need={"working_indices","target","oof_catboost","oof_xgboost","oof_lightgbm"}
    if not need.issubset(z.files): raise RuntimeError("Stage 3 NPZ не содержит обязательных полей.")
    z3={k:np.asarray(z[k]) for k in need}
if any(len(z3[k])!=289614 for k in need) or not np.array_equal(z3["working_indices"],work_i) or not np.array_equal(z3["target"],y): raise RuntimeError("Stage 3 OOF не совпадает с accepted split или target.")
for name,key in [("CatBoost","oof_catboost"),("XGBoost","oof_xgboost"),("LightGBM","oof_lightgbm")]:
    if not np.isfinite(z3[key]).all(): raise RuntimeError(f"Stage 3 OOF {name} содержит нечисловые вероятности.")
    получено,ожидается=metrics(y,z3[key]),stage1["модели"][name]["итоговые_метрики"]
    for метрика in ("ROC-AUC","Gini","PR-AUC","Precision","Recall","F1"):
        if not np.isclose(float(получено[метрика]),float(ожидается[метрика]),atol=1e-12,rtol=0):
            raise RuntimeError(f"Stage 3 OOF {name}: метрика {метрика} не совпала со Stage 1 ({получено[метрика]:.12f} != {ожидается[метрика]:.12f}). Запуск остановлен.")
evidence=json.loads(ST3J.read_text(encoding="utf-8"))
if evidence["split"]["sha256_индексов_работа"]!=WORK_SHA or evidence["split"]["final_test_использован"] or evidence["split"]["число_фолдов"]!=3 or evidence["split"]["seed"]!=42: raise RuntimeError("Stage 3 provenance не подтверждён.")
pc,px,pl=[z3[k].astype(np.float32) for k in ("oof_catboost","oof_xgboost","oof_lightgbm")]; mean=(pc+px+pl)/3
xgb_m,mean_m=metrics(y,px),metrics(y,mean); BSTAR,BMET=("GBDT_mean",mean_m) if mean_m["Gini"]>xgb_m["Gini"] else ("XGBoost",xgb_m)
FOLDS=list(StratifiedKFold(n_splits=3,shuffle=True,random_state=42).split(X,y))
display(pd.DataFrame([{"Контроль":"XGBoost",**xgb_m},{"Контроль":"GBDT_mean",**mean_m}]))
print("Preflight пройден. До TabM выбран B*:",BSTAR)

,Контроль,ROC-AUC,Gini,PR-AUC,Precision,Recall,F1,TN,FP,FN,TP
0,XGBoost,0.901995,0.803990,0.599273,0.712255,0.368445,0.485661,257429,4170,17693,10322
1,GBDT_mean,0.903200,0.806399,0.603857,0.724625,0.364162,0.484725,257722,3877,17813,10202


Preflight пройден. До TabM выбран B*: GBDT_mean


## 3. Заранее выбираем контроль для сравнения

До просмотра результата TabM сравниваем два допустимых контроля:

| Контроль | Gini | ROC-AUC | PR-AUC | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| XGBoost | 0.803990 | 0.901995 | 0.599273 | 0.712255 | 0.368445 | 0.485661 |
| Среднее CatBoost + XGBoost + LightGBM | **0.806399** | **0.903200** | **0.603857** | **0.724625** | 0.364162 | 0.484725 |

По заранее зафиксированному правилу выбирается модель с большим OOF Gini.

Поэтому контроль `B*` для Stage 7:

**`GBDT_mean` — простое арифметическое среднее вероятностей CatBoost, XGBoost и LightGBM.**

Это решение принято до просмотра результата TabM.

Подбор весов между бустинговыми моделями запрещён.

### Почему это важно?

Сравнивать новый гибрид только с XGBoost было бы недостаточно: простое среднее трёх GBDT уже оказалось немного сильнее.

Поэтому TabM должна доказать дополнительную ценность относительно **лучшего заранее определённого контроля**, а не относительно более удобного для неё baseline.

## 4. Строим признаки для TabM без утечки данных

### Что проверяем?

Для каждого внешнего фолда строятся три дополнительных признака:

- вероятность CatBoost;
- вероятность XGBoost;
- вероятность LightGBM.

Для обучающей части внешнего фолда вероятности формируются через дополнительный внутренний 3-fold cross-fitting.

Для внешней validation-части используются только модели, обученные на соответствующей train-части.

### Зачем нужна такая схема?

TabM не должна получать прогнозы от GBDT-моделей, которые уже обучались на тех же строках.

Каждая строка должна получать прогноз только от модели, которая её не видела при обучении.

Это необходимо, чтобы не допустить утечку целевой информации и искусственное завышение качества stacking-модели.

### Как это отвечает на исследовательский вопрос?

Так мы проверяем реальную дополнительную ценность прогнозов CatBoost, XGBoost и LightGBM для TabM, а не эффект утечки данных.

### Что остаётся неизменным?

Конфигурации CatBoost, XGBoost и LightGBM, схема early stopping и последующий refit соответствуют зафиксированному протоколу предыдущих этапов.

In [11]:
STATE=None; START=None; PANEL=None

def atomic(x,p=CKPT):

    q=p.with_name(p.name+".tmp"); torch.save(x,q); os.replace(q,p)

def накопить_время(fold,stage):

    now=time.perf_counter(); delta=now-STATE["последняя_точка_сессии"]

    STATE["накоплено_до_сессии"]+=delta; STATE["последняя_точка_сессии"]=now

    STATE["время_фолдов"].setdefault(str(fold),0.0); STATE["время_фолдов"][str(fold)]+=delta

    STATE["время_стадий"].setdefault(stage,0.0); STATE["время_стадий"][stage]+=delta

def total_runtime():

    return STATE["накоплено_до_сессии"]+(time.perf_counter()-STATE["последняя_точка_сессии"])

def fold_runtime(fold):

    return STATE["время_фолдов"].get(str(fold),0.0)+(time.perf_counter()-STATE["последняя_точка_сессии"] if STATE.get("active_fold")==fold else 0.0)

def panel(stage,fold,unit,done,total,best=None,auc=None,started=None):

    global PANEL

    session_start=START
    session_elapsed=0.0 if session_start is None else time.perf_counter()-session_start

    pct="—" if not total else f"{done}/{total} ({100*done/total:.0f}%)"; avg=np.mean(STATE["times"]) if STATE["times"] else None

    eta="недостаточно данных для оценки" if avg is None or not total else f"оценка: {(total-done)*avg/60:.1f} мин"

    html=HTML(f"<div style='font-family:Arial;border:1px solid #aaa;padding:10px'><h3>Этап 7 V1</h3><b>Внешний фолд:</b> {fold}/3<br><b>Крупный этап:</b> {stage}<br><b>Текущая единица:</b> {unit}<br><b>Эпоха / max:</b> {best or '—'}; <b>Лучший ROC-AUC:</b> {auc if auc is not None else 'пока нет'}<br><b>Atomic units:</b> {pct}<br><b>Время операции:</b> {0 if started is None else time.perf_counter()-started:.1f} сек; <b>фолда:</b> {fold_runtime(fold)/60:.1f} мин; <b>сессии:</b> {session_elapsed/60:.1f} мин; <b>общее:</b> {total_runtime()/60:.1f} мин<br><b>Среднее сопоставимой операции:</b> {'пока нет' if avg is None else f'{avg:.1f} сек'}; <b>Осталось:</b> {eta}</div>")

    if PANEL is None:PANEL=display(html,display_id=True)

    else:PANEL.update(html)

def save(stage,fold,unit,started):

    STATE["times"].append(time.perf_counter()-started); накопить_время(fold,stage); atomic(STATE); panel(stage,fold,unit,len(STATE["done"]),None,started=started)

def gbdt_predict(name: str, tr: np.ndarray, va: np.ndarray, seed: int) -> np.ndarray:

    """Stage 1: 90/10 early stopping, best iteration, новый estimator и refit."""

    a,b=train_test_split(np.arange(len(tr)),test_size=.1,stratify=y[tr],random_state=seed)

    if name=="CatBoost":

        search=CatBoostClassifier(**GBDT[name],random_seed=seed)

        search.fit(X[tr[a]],y[tr[a]],eval_set=(X[tr[b]],y[tr[b]]),early_stopping_rounds=40,verbose=False)

        best=search.get_best_iteration(); n=int(best)+1 if best is not None and best>=0 else 900

        fitted=CatBoostClassifier(**{**GBDT[name],"iterations":n},random_seed=seed)

        fitted.fit(X[tr],y[tr]); probabilities=np.asarray(fitted.predict_proba(X[va]),dtype=np.float32)

        return probabilities[:,1]

    if name=="XGBoost":

        search=XGBClassifier(**GBDT[name],random_state=seed,early_stopping_rounds=40)

        search.fit(X[tr[a]],y[tr[a]],eval_set=[(X[tr[b]],y[tr[b]])],verbose=False)

        best=getattr(search,"best_iteration",None); n=int(best)+1 if best is not None else 900

        fitted=XGBClassifier(**{**GBDT[name],"n_estimators":n},random_state=seed)

        fitted.fit(X[tr],y[tr]); probabilities=np.asarray(fitted.predict_proba(X[va]),dtype=np.float32)

        return probabilities[:,1]

    if name=="LightGBM":

        search=LGBMClassifier(**GBDT[name],random_state=seed)

        search.fit(X[tr[a]],y[tr[a]],eval_set=[(X[tr[b]],y[tr[b]])],callbacks=[early_stopping(40,verbose=False)])

        best=getattr(search,"best_iteration_",None); n=int(best) if best is not None and best>0 else 900

        fitted=LGBMClassifier(**{**GBDT[name],"n_estimators":n},random_state=seed)

        fitted.fit(X[tr],y[tr]); probabilities=np.asarray(fitted.predict_proba(X[va]),dtype=np.float32)

        return probabilities[:,1]

    raise KeyError(f"Неизвестная GBDT-модель: {name}")

def crossfit(indices,fold,label):

    key=label+":p"; STATE["arrays"].setdefault(key,np.full((len(indices),3),np.nan,np.float32)); out=STATE["arrays"][key]

    for j,(a,b) in enumerate(StratifiedKFold(n_splits=3,shuffle=True,random_state=FSEED[fold]).split(indices,y[indices]),1):

        for c,name in enumerate(("CatBoost","XGBoost","LightGBM")):

            op=f"{label}:{j}:{name}"

            if op in STATE["done"]: continue

            t=time.perf_counter();panel("построение nested GBDT",fold,f"{name}; inner fold {j}/3",len(STATE["done"]),None,started=t)

            out[b,c]=gbdt_predict(name,indices[a],indices[b],FSEED[fold]);STATE["done"].add(op);save("построение nested GBDT",fold,op,t)

    if not np.isfinite(out).all(): raise RuntimeError("Незавершённый nested cross-fitting.")

    return out

def holdout(tr,va,fold,label):

    key=label+":p"; STATE["arrays"].setdefault(key,np.full((len(va),3),np.nan,np.float32));out=STATE["arrays"][key]

    for c,name in enumerate(("CatBoost","XGBoost","LightGBM")):

        op=f"{label}:{name}"

        if op in STATE["done"]:continue

        t=time.perf_counter();panel("подготовка E_f",fold,name,len(STATE["done"]),None,started=t)

        out[:,c]=gbdt_predict(name,tr,va,FSEED[fold]);STATE["done"].add(op);save("подготовка E_f",fold,op,t)

    return out

## 5. Обучаем TabM второго уровня и защищаем длинный запуск

### Что проверяем?

Для каждого внешнего фолда TabM получает:

**47 исходных признаков + 3 leakage-safe GBDT-вероятности = 50 входных признаков.**

Число эпох определяется только на внутренней validation-части соответствующего train-фолда.

После выбора лучшей эпохи создаётся новая TabM и заново обучается на полном train-фолде.

### Что важно для честности эксперимента?

Внешний validation-фолд не участвует:

- в выборе числа эпох;
- в early stopping;
- в подборе параметров;
- в обучении GBDT-признаков.

Он используется только для внешней OOF-оценки.

### Checkpoint и продолжение после остановки

Длинный CPU-run защищён механизмом `checkpoint/resume`.

Сохраняются:

- состояние модели;
- состояние optimizer;
- состояния генераторов случайных чисел;
- состояние DataLoader;
- завершённые GBDT-операции;
- уже рассчитанные результаты фолдов.

После аварийной остановки повторно выполняется только незавершённая операция.

Checkpoint/resume **не меняет ML-протокол и не влияет на результат эксперимента**.

In [4]:
def seed(s): random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.use_deterministic_algorithms(True)
def net(): return tabm.TabM.make(n_num_features=50,cat_cardinalities=None,arch_type=CFG["arch_type"],k=8,n_blocks=3,d_block=512,activation="ReLU",dropout=.1,start_scaling_init="random-signs",num_embeddings=None,d_out=2)
def make_loader(x,yy,s,g=None):
    gen=torch.Generator(device="cpu");gen.manual_seed(s)
    if g is not None:gen.set_state(g)
    return DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(yy.astype(np.int64))),batch_size=256,shuffle=True,generator=gen,num_workers=0),gen
def epoch(m,l,o):
    m.train()
    for xb,yb in l:
        o.zero_grad(set_to_none=True); z=m(xb.float()); F.cross_entropy(z.reshape(-1,2),yb[:,None].expand(-1,m.k).reshape(-1)).backward();torch.nn.utils.clip_grad_norm_(m.parameters(),1.);o.step()
@torch.inference_mode()
def pred(m,x):
    m.eval();t=torch.from_numpy(x).float();return np.concatenate([torch.softmax(m(t[i:i+256]),dim=-1).mean(1)[:,1].cpu().numpy() for i in range(0,len(t),256)]).astype(np.float32)
def restore(r): random.setstate(r["python"]);np.random.set_state(r["numpy"]);torch.set_rng_state(r["torch"])
def train_tabm(kind,xt,yt,xv,yv,fold,epochs,select=False):
    r=STATE["tabm"].setdefault(f"{fold}:{kind}",{"epoch":0,"best":0,"auc":-np.inf,"stale":0})
    seed(FSEED[fold]);m=net();o=torch.optim.AdamW(m.parameters(),lr=.002,weight_decay=.0003,betas=(.9,.999),eps=1e-8)
    if r["epoch"]:m.load_state_dict(r["model"]);o.load_state_dict(r["optimizer"]);restore(r["rng"])
    l,g=make_loader(xt,yt,FSEED[fold],r.get("generator"))
    for e in range(r["epoch"]+1,epochs+1):
        t=time.perf_counter();panel("выбор epochs TabM" if select else "final TabM refit",fold,f"эпоха {e}/{epochs}",e-1,epochs,r.get("best"),None if not np.isfinite(r.get("auc",-np.inf)) else f"{r['auc']:.5f}",t);epoch(m,l,o)
        if select:
            a=float(roc_auc_score(yv,pred(m,xv)))
            if a>r["auc"]:r.update({"best":e,"auc":a,"stale":0})
            else:r["stale"]+=1
        r.update({"epoch":e,"model":m.state_dict(),"optimizer":o.state_dict(),"generator":g.get_state(),"rng":{"python":random.getstate(),"numpy":np.random.get_state(),"torch":torch.get_rng_state()}})
        STATE["done"].add(f"{fold}:{kind}:epoch:{e}");save("выбор epochs TabM" if select else "final TabM refit",fold,f"эпоха {e}",t)
        if select and r["stale"]>=16:break
    return (int(r["best"]),float(r["auc"])) if select else pred(m,xv)
def contract():
    return {
        "experiment":"Stage 7","version":"V1","dataset_sha256":DATA_SHA,"working_index_sha256":WORK_SHA,
        "raw_features_in_order":FEATURES,"raw_feature_count":47,
        "probability_features_in_order":["p_catboost","p_xgboost","p_lightgbm"],"probability_feature_count":3,"d_in":50,
        "forbidden_features":FORBIDDEN,"target":TARGET,"identifier":IDENTIFIER,
        "outer_cv":{"type":"StratifiedKFold","n_splits":3,"shuffle":True,"random_state":42},"outer_fold_seeds":FSEED,
        "stage3_oof":{"path":ST3.name,"sha256":sha(ST3),"provenance_sha256":sha(ST3J),"usage":"только соответствующий outer validation-фолд V_f"},
        "gbdt_training_contract":{
            "parameters":GBDT,"inner_validation_protocol":"stratified 90/10 split внутри соответствующего train; затем refit на всём train",
            "inner_train_fraction":.90,"inner_validation_fraction":.10,"early_stopping_rounds":40,
            "selection_metric_and_mechanism":{"CatBoost":"eval_metric=AUC; get_best_iteration","XGBoost":"eval_metric=logloss; best_iteration","LightGBM":"early_stopping callback; best_iteration_"},
            "refit_rule":"best iteration -> новый estimator -> refit на полном соответствующем train",
            "nested_stacking_cv":{"type":"StratifiedKFold","n_splits":3,"shuffle":True,"random_state":"outer fold seed"},
            "outer_cv":{"type":"StratifiedKFold","n_splits":3,"shuffle":True,"random_state":42},"seeds":FSEED},
        "tabm_training_contract":{
            "arch_type":"tabm","k":8,"n_blocks":3,"d_block":512,"activation":"ReLU","dropout":.1,"start_scaling_init":"random-signs","num_embeddings":None,"d_out":2,
            "input_dtype":"float32","optimizer":"AdamW","lr":.002,"weight_decay":.0003,"betas":(.9,.999),"eps":1e-8,"gradient_clip_global_norm":1.,"batch_size":256,
            "share_training_batches":True,"max_epochs":100,"amp":False,"torch_compile":False,"scheduler":None,"warmup":None,"class_weights":None,"sampling":None,"d_in":50,
            "selection_early_stopping":{"inner_train_fraction":.90,"inner_validation_fraction":.10,"patience":16,"selection_metric":"ROC-AUC","refit_epochs":"best_epoch"},"outer_fold_seeds":FSEED},
        "baseline_contract":{"B_star_selection_rule":"больший OOF Gini из XGBoost и GBDT_mean до просмотра TabM","GBDT_mean":"простое арифметическое среднее p_catboost, p_xgboost, p_lightgbm","weighted_averaging_forbidden":True},
        "final_test_used":False
    }

## 6. Собираем полный контролируемый эксперимент

### Что проверяем?

Следующая code-ячейка собирает единый сценарий полного Stage 7.

Для каждого из трёх внешних фолдов она должна:

1. построить leakage-safe прогнозы CatBoost, XGBoost и LightGBM;
2. добавить эти три прогноза к 47 исходным признакам;
3. выбрать число эпох TabM только внутри обучающей части фолда;
4. заново обучить TabM на полном train-фолде;
5. получить прогноз для внешнего validation-фолда;
6. собрать единый OOF-прогноз после завершения всех трёх фолдов;
7. сравнить результат с заранее выбранным контролем `B* = GBDT_mean`;
8. сохранить итоговые результаты и артефакты эксперимента.

### Зачем?

До этого момента мы только подготовили отдельные компоненты эксперимента.

Теперь они объединяются в один воспроизводимый controlled experiment.

### Как принимается решение?

Результат гибридной TabM сравнивается с заранее выбранным `B* = GBDT_mean`.

Правила сравнения были зафиксированы до просмотра результата TabM, поэтому итоговое решение не подстраивается под полученные метрики.

### Что остаётся неизменным?

Не выполняются:

- подбор гиперпараметров;
- балансировка классов;
- подбор весов классов;
- калибровка;
- подбор порога классификации;
- изменение folds;
- изменение seeds;
- использование финальной тестовой выборки.

Следующая code-ячейка только определяет функцию `run_stage7()`. Полный дорогой запуск на этом этапе ещё не начинается.

In [12]:
def write_json(x,p):
    p.parent.mkdir(parents=True,exist_ok=True);q=p.with_name(p.name+".tmp");q.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding="utf-8");os.replace(q,p)
def run_stage7():
    global STATE,START,PANEL
    START=time.perf_counter();PANEL=None;c=contract()
    if CKPT.exists():
        try:STATE=torch.load(CKPT,map_location="cpu",weights_only=False)
        except TypeError:STATE=torch.load(CKPT,map_location="cpu")
        if STATE["contract"]!=c:raise RuntimeError("Checkpoint несовместим с experiment/data/features/config identity.")
        STATE.setdefault("накоплено_до_сессии",STATE.pop("runtime",0.0));STATE.setdefault("время_фолдов",{});STATE.setdefault("время_стадий",{});STATE["последняя_точка_сессии"]=START
        print("Resume: downtime между сессиями исключён; готовые операции не пересчитываются.")
    else:
        STATE={"contract":c,"done":set(),"arrays":{},"tabm":{},"rows":[],"times":[],"накоплено_до_сессии":0.0,"время_фолдов":{},"время_стадий":{},"active_fold":None,"последняя_точка_сессии":START};atomic(STATE)
    try:
        for fold,(tr,va) in enumerate(FOLDS,1):
            if any(r["fold"]==fold for r in STATE["rows"]):continue
            STATE["active_fold"]=fold;ft=time.perf_counter();A,E=train_test_split(tr,test_size=.1,stratify=y[tr],random_state=FSEED[fold])
            pa=crossfit(A,fold,f"{fold}:A");pe=holdout(A,E,fold,f"{fold}:E")
            be,ba=train_tabm("select",np.column_stack([X[A],pa]),y[A],np.column_stack([X[E],pe]),y[E],fold,100,True)
            pt=crossfit(tr,fold,f"{fold}:T");pv=np.column_stack([pc[va],px[va],pl[va]])
            p=train_tabm("refit",np.column_stack([X[tr],pt]),y[tr],np.column_stack([X[va],pv]),y[va],fold,be,False)
            bm=metrics(y[va],mean[va] if BSTAR=="GBDT_mean" else px[va]);row={"fold":fold,"seed":FSEED[fold],"best_epoch":be,"inner_best_roc_auc":ba,"runtime_seconds":0.0,"metrics":metrics(y[va],p),"baseline_fold":bm};STATE["arrays"][f"{fold}:outer"]=p;STATE["done"].add(f"{fold}:outer_completed");save("metrics",fold,"внешний фолд завершён",ft);row["runtime_seconds"]=STATE["время_фолдов"][str(fold)];STATE["rows"].append(row);atomic(STATE)
        o=np.full(len(y),np.nan,np.float32);farr=np.zeros(len(y),np.int8)
        for f,(_,va) in enumerate(FOLDS,1):o[va]=STATE["arrays"][f"{f}:outer"];farr[va]=f
        if not np.isfinite(o).all():raise RuntimeError("Неполный OOF; финальные artifacts не создаются.")
        накопить_время(3,"финальные метрики");hm=metrics(y,o);d={k:float(hm[k]-BMET[k]) for k in ("ROC-AUC","Gini","PR-AUC","Precision","Recall","F1")};wins=sum(r["metrics"]["Gini"]>r["baseline_fold"]["Gini"] for r in STATE["rows"]);loss=sum(r["metrics"]["Gini"]<r["baseline_fold"]["Gini"] for r in STATE["rows"]);dec="meaningful_improvement" if d["Gini"]>=.01 and d["PR-AUC"]>=.005 and wins>=2 else "inferior" if d["Gini"]<=-.01 and d["PR-AUC"]<=-.005 and loss>=2 else "no_material_benefit"
        res={**c,"status":"completed","target":TARGET,"identifier":IDENTIFIER,"final_test_used":False,"stage3_provenance":{"validated":True,"npz":ST3.name},"baseline_selection":{"B_star":BSTAR,"xgboost_metrics":xgb_m,"gbdt_mean_metrics":mean_m,"weighted_averaging":False},"fold_metrics":STATE["rows"],"oof_metrics":hm,"deltas_vs_B_star":d,"runtime_seconds":STATE["накоплено_до_сессии"],"runtime_breakdown_seconds":{"outer_folds":STATE["время_фолдов"],"stages":STATE["время_стадий"]},"decision":dec,"limitations":["Random CV не доказывает temporal stability.","Три folds не являются claim о statistical significance.","Порог 0.5 диагностический.","Final test не использован.","Вывод относится только к locked Stage 7 hybrid design.","Результат stacking не доказывает бизнес-пользу."]}
        summ={"experiment":"Stage 7","version":"V1","status":"completed","B_star":BSTAR,"oof_metrics":hm,"deltas_vs_B_star":d,"decision":dec,"runtime_seconds":res["runtime_seconds"],"runtime_breakdown_seconds":res["runtime_breakdown_seconds"],"limitations":res["limitations"]};G.mkdir(parents=True,exist_ok=True);np.savez_compressed(OOF,working_indices=work_i,target=y,fold=farr,p_catboost=pc,p_xgboost=px,p_lightgbm=pl,gbdt_mean=mean,tabm_oof_probability=o);write_json(res,RESULT);write_json(summ,SUMMARY);CKPT.unlink(missing_ok=True);return res
    except Exception:atomic(STATE);raise
print("Функция run_stage7() подготовлена; полный ML-run не запускается автоматически.")

Функция run_stage7() подготовлена; полный ML-run не запускается автоматически.


## 7. Проверяем корректность checkpoint/resume на маленьком тесте

### Что проверяем?

Перед дорогим запуском отдельно проверяем технический механизм сохранения и восстановления обучения.

На маленьком синтетическом примере выполняются два варианта одного и того же обучения:

- непрерывное обучение;
- обучение с остановкой, сохранением checkpoint и последующим продолжением.

После завершения сравниваются веса модели и её прогнозы.

### Зачем?

Продолжение после checkpoint считается корректным только в том случае, если приводит к тому же состоянию модели, что и непрерывный запуск.

Это особенно важно для долгого Stage 7, который может выполняться несколько часов.

### Как это отвечает на исследовательский вопрос?

Напрямую качество модели эта проверка не оценивает.

Она подтверждает, что технический механизм `checkpoint/resume` не изменяет результат эксперимента и позволяет безопасно продолжить прерванный запуск.

### Что остаётся неизменным?

Синтетический тест:

- не использует данные KOMUS;
- не создаёт исследовательские результаты Stage 7;
- не влияет на OOF-метрики;
- не меняет ML-протокол.

In [6]:
def synthetic_smoke() -> None:
    """Реальная CPU-проверка dropout, shuffled DataLoader и checkpoint/resume."""
    def seed_all(seed: int):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.use_deterministic_algorithms(True)
    def model():
        return torch.nn.Sequential(torch.nn.Linear(4, 12), torch.nn.ReLU(), torch.nn.Dropout(.25), torch.nn.Linear(12, 2))
    x=torch.arange(96,dtype=torch.float32).reshape(24,4)/97; y0=(torch.arange(24)%2).long()
    def loader_for(generator_state=None):
        g=torch.Generator(device="cpu");g.manual_seed(123)
        if generator_state is not None:g.set_state(generator_state)
        return DataLoader(TensorDataset(x,y0),batch_size=6,shuffle=True,generator=g,num_workers=0),g
    def train(m,o,l,first,last):
        for epoch in range(first,last+1):
            m.train()
            for xb,yb in l:o.zero_grad(set_to_none=True);torch.nn.functional.cross_entropy(m(xb),yb).backward();o.step()
        return epoch
    seed_all(777); cm=model();co=torch.optim.AdamW(cm.parameters(),lr=.01);cl,cg=loader_for();train(cm,co,cl,1,4);cm.eval();cp=cm(x).detach().clone();cw={k:v.detach().clone() for k,v in cm.state_dict().items()}
    seed_all(777); im=model();io=torch.optim.AdamW(im.parameters(),lr=.01);il,ig=loader_for();last=train(im,io,il,1,2)
    with tempfile.TemporaryDirectory() as d:
        path=Path(d)/"checkpoint.pt";tmp=Path(d)/"checkpoint.tmp"
        torch.save({"model":im.state_dict(),"optimizer":io.state_dict(),"python":random.getstate(),"numpy":np.random.get_state(),"torch":torch.get_rng_state(),"generator":ig.get_state(),"last_epoch":last},tmp);os.replace(tmp,path)
        saved=torch.load(path,map_location="cpu",weights_only=False)
    rm=model();ro=torch.optim.AdamW(rm.parameters(),lr=.01);rm.load_state_dict(saved["model"]);ro.load_state_dict(saved["optimizer"]);random.setstate(saved["python"]);np.random.set_state(saved["numpy"]);torch.set_rng_state(saved["torch"]);rl,rg=loader_for(saved["generator"]);resumed=train(rm,ro,rl,saved["last_epoch"]+1,4);rm.eval();rp=rm(x).detach()
    wd=max(float((cw[k]-v).abs().max()) for k,v in rm.state_dict().items());pd=float((cp-rp).abs().max())
    assert resumed==saved["last_epoch"]+2 and wd==0.0 and pd==0.0, (resumed,wd,pd)
    print(f"Smoke пройден: resume начат с эпохи {saved['last_epoch']+1}; max |Δ weights|={wd:.1f}; max |Δ predictions|={pd:.1f}.")
synthetic_smoke()

Smoke пройден: resume начат с эпохи 3; max |Δ weights|=0.0; max |Δ predictions|=0.0.


## 8. Подготавливаем формирование итогового результата

### Что проверяем?

Следующая техническая ячейка определяет функцию, которая после завершения Stage 7 читает сохранённый итоговый артефакт и формирует краткое описание фактического результата.

### Зачем?

Итоговые выводы должны строиться только из реально сохранённых метрик эксперимента, а не задаваться заранее.

### Как это отвечает на исследовательский вопрос?

После полного запуска функция сможет показать:

- выбранный контроль `B*`;
- итоговые OOF-метрики гибридной модели;
- разницу относительно `B*`;
- время выполнения;
- зафиксированное решение эксперимента.

### Что остаётся неизменным?

Следующая code-ячейка не обучает модели, не изменяет результаты и не запускает Stage 7.

Она только подготавливает человекочитаемый вывод из уже сохранённых фактических результатов.

In [10]:
def final_section():
    if not SUMMARY.exists():
        display(Markdown("### FACTS\nПолный запуск ещё не выполнен; финальные OOF metrics отсутствуют."));return
    s=json.loads(SUMMARY.read_text(encoding="utf-8"));d=s["decision"]
    text={"meaningful_improvement":"Да: locked hybrid design дал meaningful improvement относительно B*. Следующий вопрос: воспроизводится ли эффект на заранее закрытом final test по отдельному protocol?","inferior":"Нет: locked hybrid design уступил B*. Следующий вопрос: нужен ли другой современный controlled approach?","no_material_benefit":"Нет подтверждённой материальной дополнительной пользы. Следующий вопрос: есть ли иной современный controlled approach с независимым сигналом?"}[d]
    display(Markdown(f"## Результат исследования\n### FACTS\n- B*: **{s['B_star']}**.\n- Hybrid OOF Gini: **{s['oof_metrics']['Gini']:.6f}**; PR-AUC: **{s['oof_metrics']['PR-AUC']:.6f}**.\n- Delta Gini: **{s['deltas_vs_B_star']['Gini']:+.6f}**; Delta PR-AUC: **{s['deltas_vs_B_star']['PR-AUC']:+.6f}**.\n- Runtime: **{s['runtime_seconds']/60:.1f} мин**.\n### INTERPRETATION\n{text}\n### LIMITATIONS\n- Random CV не доказывает temporal stability.\n- Три folds не являются claim о statistical significance.\n- Порог 0.5 диагностический; final test не использован.\n- Результат относится только к locked Stage 7 hybrid design.\n- Результат stacking не доказывает бизнес-пользу.\n### NEXT STEP\nСформулирован в интерпретации по фактическому решению."))
final_section()

## Результат исследования
### FACTS
- B*: **GBDT_mean**.
- Hybrid OOF Gini: **0.804260**; PR-AUC: **0.605662**.
- Delta Gini: **-0.002139**; Delta PR-AUC: **+0.001805**.
- Runtime: **345.0 мин**.
### INTERPRETATION
Нет подтверждённой материальной дополнительной пользы. Следующий вопрос: есть ли иной современный controlled approach с независимым сигналом?
### LIMITATIONS
- Random CV не доказывает temporal stability.
- Три folds не являются claim о statistical significance.
- Порог 0.5 диагностический; final test не использован.
- Результат относится только к locked Stage 7 hybrid design.
- Результат stacking не доказывает бизнес-пользу.
### NEXT STEP
Сформулирован в интерпретации по фактическому решению.

## 9. Запускаем полный Stage 7 V1

### Что запускаем?

Теперь выполняется полный 3-fold OOF-эксперимент:

**47 исходных признаков + leakage-safe вероятности CatBoost, XGBoost и LightGBM → TabM.**

### Что уже проверено перед запуском?

К этому моменту подтверждены:

- контрольная сумма и идентичность датасета;
- рабочая выборка;
- 47 разрешённых исходных признаков;
- отсутствие `Q_B1_norm` и `Q_B2_norm` среди predictors;
- происхождение сохранённых GBDT OOF-прогнозов;
- контроль `B* = GBDT_mean`;
- leakage-safe схема построения stacking-признаков;
- конфигурация TabM;
- корректность механизма `checkpoint/resume`.

### Как проходит эксперимент?

Для каждого внешнего фолда независимо:

1. строятся три GBDT-признака;
2. выбирается число эпох TabM только внутри train-части;
3. TabM переобучается на полном train-фолде;
4. строится прогноз для внешнего validation-фолда.

После завершения трёх фолдов формируется единый OOF-прогноз и рассчитываются итоговые метрики.

### Как работает восстановление после остановки?

Если совместимый checkpoint существует, расчёт продолжается с первой незавершённой операции.

Уже завершённые операции повторно не выполняются.

Если checkpoint отсутствует, эксперимент начинается с нуля.

### Что остаётся неизменным?

Финальная тестовая выборка не используется.

Подбор параметров, балансировка классов, калибровка и подбор порога не выполняются.

Время выполнения измеряется как характеристика эксперимента, но жёсткого ограничения по runtime нет.

In [8]:
# Полный запуск Stage 7 V1.
# Эта ячейка не меняет experiment contract:
# она только вызывает уже определённый run_stage7().

if CKPT.exists():
    display(
        Markdown(
            "### ▶ Stage 7 V1 — восстановление\n"
            "Найден совместимый checkpoint-кандидат. "
            "Перед продолжением `run_stage7()` проверит experiment contract "
            "и возобновит расчёт только при полном совпадении."
        )
    )
else:
    display(
        Markdown(
            "### ▶ Stage 7 V1 — запуск с нуля\n"
            "Checkpoint отсутствует. Начинается новый полный 3-fold run."
        )
    )

try:
    stage7_result = run_stage7()

except KeyboardInterrupt:
    display(
        Markdown(
            "### ⏸ Stage 7 V1 прерван пользователем\n"
            "Финальный результат не объявляется. "
            "Последний успешно записанный checkpoint сохранён. "
            "После повторного запуска этой же ячейки эксперимент продолжится "
            "с первой незавершённой операции."
        )
    )
    raise

except Exception as exc:
    display(
        Markdown(
            "### ❌ Stage 7 V1 остановлен из-за ошибки\n"
            f"**Тип ошибки:** `{type(exc).__name__}`  \n"
            f"**Сообщение:** `{exc}`  \n\n"
            "Финальные метрики не интерпретируются. "
            "Если был создан checkpoint, он не удаляется."
        )
    )
    raise

else:
    display(
        Markdown(
            "### ✅ Stage 7 V1 завершён\n"
            "Полный 3-fold OOF-run завершился штатно. "
            "Ниже формируется человекочитаемый итог только из сохранённых "
            "фактических результатов."
        )
    )
    final_section()

### ▶ Stage 7 V1 — запуск с нуля
Checkpoint отсутствует. Начинается новый полный 3-fold run.

d:\Projects\komus-credit-risk\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Projects\komus-credit-risk\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Projects\komus-credit-risk\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Projects\komus-credit-risk\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(e

### ✅ Stage 7 V1 завершён
Полный 3-fold OOF-run завершился штатно. Ниже формируется человекочитаемый итог только из сохранённых фактических результатов.

## Результат исследования
### FACTS
- B*: **GBDT_mean**.
- Hybrid OOF Gini: **0.804260**; PR-AUC: **0.605662**.
- Delta Gini: **-0.002139**; Delta PR-AUC: **+0.001805**.
- Runtime: **345.0 мин**.
### INTERPRETATION
Нет подтверждённой материальной дополнительной пользы. Следующий вопрос: есть ли иной современный controlled approach с независимым сигналом?
### LIMITATIONS
- Random CV не доказывает temporal stability.
- Три folds не являются claim о statistical significance.
- Порог 0.5 диагностический; final test не использован.
- Результат относится только к locked Stage 7 hybrid design.
- Результат stacking не доказывает бизнес-пользу.
### NEXT STEP
Сформулирован в интерпретации по фактическому решению.

# Результат исследования

## ФАКТЫ

Полный Stage 7 V1 успешно завершён.

Проверена гибридная схема:

**47 исходных разрешённых признаков + leakage-safe вероятности CatBoost, XGBoost и LightGBM → TabM.**

Полный OOF-прогноз построен для всех `289 614` наблюдений рабочей выборки.

### Итоговые результаты TabM поверх GBDT

| Метрика | Результат |
|---|---:|
| Gini | **0.804260** |
| ROC-AUC | **0.902130** |
| PR-AUC | **0.605662** |
| Precision | **0.719060** |
| Recall | **0.375585** |
| F1 | **0.493435** |

При диагностическом пороге `0.5`:

- верно определено недефолтов (`TN`) — `257 488`;
- ложных срабатываний (`FP`) — `4 111`;
- пропущенных дефолтов (`FN`) — `17 493`;
- найденных дефолтов (`TP`) — `10 522`.

### Результаты по отдельным внешним фолдам

| Фолд | Gini | ROC-AUC | PR-AUC | Precision | Recall | F1 |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0.799354 | 0.899677 | 0.602232 | 0.723080 | 0.371064 | 0.490446 |
| 2 | 0.807300 | 0.903650 | 0.603437 | 0.729325 | 0.359820 | 0.481893 |
| 3 | 0.810610 | 0.905305 | 0.613153 | 0.706343 | 0.395867 | 0.507377 |

Лучшее число эпох TabM было выбрано только на внутренних validation-частях:

- фолд 1 — `27` эпох;
- фолд 2 — `21` эпоха;
- фолд 3 — `23` эпохи.

### Контрольная модель

До просмотра результата TabM был заранее выбран контроль:

**`GBDT_mean` — простое среднее вероятностей CatBoost, XGBoost и LightGBM.**

Его OOF-метрики:

| Метрика | GBDT_mean |
|---|---:|
| Gini | **0.806399** |
| ROC-AUC | **0.903200** |
| PR-AUC | **0.603857** |
| Precision | **0.724625** |
| Recall | **0.364162** |
| F1 | **0.484725** |

### Разница TabM поверх GBDT относительно GBDT_mean

| Метрика | Разница |
|---|---:|
| Gini | **-0.002139** |
| ROC-AUC | **-0.001070** |
| PR-AUC | **+0.001805** |
| Precision | **-0.005566** |
| Recall | **+0.011422** |
| F1 | **+0.008710** |

При пороге `0.5` гибридная модель по сравнению с `GBDT_mean`:

- нашла на `320` дефолтов больше;
- допустила на `320` меньше пропущенных дефолтов;
- одновременно получила на `234` больше ложноположительных срабатываний.

### Сравнение с самостоятельной TabM из Stage 6

Самостоятельная TabM имела:

- `Gini = 0.781310`;
- `ROC-AUC = 0.890655`;
- `PR-AUC = 0.567993`;
- `Precision = 0.737602`;
- `Recall = 0.280850`;
- `F1 = 0.406804`.

После добавления прогнозов трёх GBDT изменения составили:

- `Gini: +0.022950`;
- `ROC-AUC: +0.011475`;
- `PR-AUC: +0.037669`;
- `Precision: -0.018542`;
- `Recall: +0.094735`;
- `F1: +0.086631`.

### Время выполнения

Полный Stage 7 занял:

**`20 700.83 сек` ≈ `345.0 мин` ≈ `5 ч 45 мин`.**

Основные затраты времени:

- выбор числа эпох TabM — около `191.8 мин`;
- финальное переобучение TabM — около `126.2 мин`;
- построение nested GBDT — около `21.1 мин`;
- подготовка внешних GBDT-признаков — около `5.4 мин`.

Финальная тестовая выборка не использовалась.

## ИНТЕРПРЕТАЦИЯ

Исследовательский вопрос Stage 7 получил ответ.

**Использование прогнозов CatBoost, XGBoost и LightGBM как дополнительных признаков действительно существенно улучшило результат TabM по сравнению с её самостоятельной версией.**

Особенно заметно выросли:

- `Recall`;
- `F1`;
- `PR-AUC`;
- `Gini`.

Это показывает, что TabM смогла использовать информацию, уже извлечённую бустинговыми моделями, намного эффективнее, чем только исходные 47 признаков.

Однако основной вопрос Stage 7 заключался не в том, станет ли TabM лучше самой себя, а в том, **даст ли гибридная схема дополнительную ценность поверх уже сильного GBDT-решения**.

Здесь результат оказался смешанным.

Относительно заранее выбранного `GBDT_mean`:

- `Gini` немного снизился;
- `ROC-AUC` немного снизился;
- `Precision` немного снизился;
- `PR-AUC` немного вырос;
- `Recall` вырос;
- `F1` вырос.

То есть гибридная модель стала немного чаще находить дефолты, но одновременно стала чаще ошибочно относить недефолтные компании к рискованным.

По основной ранжирующей метрике `Gini` преимущества над `GBDT_mean` нет.

Кроме того, на каждом из трёх внешних фолдов Gini гибридной TabM был немного ниже соответствующего Gini `GBDT_mean`.

По заранее зафиксированному правилу итог Stage 7:

**подтверждённого материального преимущества TabM поверх сильного GBDT baseline не получено.**

При этом нельзя сказать, что эксперимент оказался неудачным.

Он показал важную вещь: **stacking действительно сильно улучшает саму TabM, но этого улучшения недостаточно, чтобы убедительно превзойти уже сильную простую комбинацию бустинговых моделей.**

## ОГРАНИЧЕНИЯ

- Случайная кросс-валидация и OOF-оценка не доказывают временную устойчивость модели.
- Три внешних фолда сами по себе не доказывают статистическую значимость небольших различий между моделями.
- Порог `0.5` использовался только как диагностический и специально для Stage 7 не оптимизировался.
- Поэтому увеличение `Recall` и `F1` при этом пороге нельзя автоматически считать доказанной бизнес-пользой.
- Финальная тестовая выборка не использовалась.
- Веса CatBoost, XGBoost и LightGBM не подбирались: `GBDT_mean` — простое арифметическое среднее их вероятностей.
- Дополнительный подбор гиперпараметров TabM специально под stacking-признаки не проводился.
- Результат относится только к зафиксированной архитектуре Stage 7: 47 исходных признаков + три leakage-safe GBDT-вероятности → TabM.
- Эксперимент не доказывает, что другие способы stacking или другие современные подходы не могут дать дополнительный независимый сигнал.
- Полученные зависимости не являются доказательством причинного влияния признаков на дефолт.

## СЛЕДУЮЩИЙ ШАГ

Этапы 6 и 7 вместе дают достаточно ясный ответ по текущей ветке TabM.

### Что мы узнали?

1. **Самостоятельная TabM** на 47 разрешённых признаках заметно уступила сильным бустинговым моделям.
2. **TabM поверх прогнозов CatBoost, XGBoost и LightGBM** существенно улучшила результат самой TabM.
3. Однако относительно заранее выбранного сильного контроля `GBDT_mean` подтверждённого материального преимущества не получено.
4. Дополнительная сложность stacking-схемы в текущем виде не подтверждена достаточным выигрышем по основной метрике `Gini`.

Поэтому продолжать усложнять именно эту ветку TabM без нового исследовательского основания сейчас нецелесообразно.

### Следующий исследовательский вопрос

**Может ли другой современный подход, отсутствующий в предыдущем исследовании KOMUS, дать дополнительный независимый OOF-сигнал на тех же 47 разрешённых признаках при сохранении текущего протокола сравнения?**

Выбор следующего подхода должен быть зафиксирован отдельно до просмотра его результатов.

Финальная тестовая выборка остаётся закрытой.